In [ ]:
import re
import numpy as np
import pandas as pd

# -----------------------------
# CONFIG / INPUTS
# -----------------------------
INPUT_MUTATIONS = ['P04637', 'R175D', 'P04637', 'R175S']
BASE_DIR = "../medha"
OUT_CSV = "../complete_sequence_dataset_features.csv"

MOTIFS = ['nM', 'Mc', 'n_M', 'M_c', 'n__M', 'M__c', 'tri']

AACON_HEADER = [
    'mut_pos', 'KABAT', 'JORES', 'SCHNEIDER', 'SHENKIN', 'GERSTEIN',
    'TAYLOR_GAPS', 'TAYLOR_NO_GAPS', 'VELIBIL', 'KARLIN', 'ARMON',
    'THOMPSON', 'NOT_LANCET', 'MIRNY', 'WILLIAMSON', 'LANDGRAF',
    'SANDER', 'VALDAR', 'SMERFS'
]

# -----------------------------
# LOAD REFERENCE TABLES
# -----------------------------
uniprot_seq = pd.read_excel(f"{BASE_DIR}/whole_proteome_uniprot_id.xlsx", engine="openpyxl")
uniprot_seq.rename({"From": "uniprot_ids"}, axis=1, inplace=True)
uniprot_seq["gene"] = uniprot_seq["Entry Name"].apply(lambda x: x.split("_")[0])

physchem_norm = pd.read_csv(f"{BASE_DIR}/49_properties_normalizedValues.csv")
physchem_num = pd.read_csv(f"{BASE_DIR}/49_properties_numerical_Values.csv")
extra_463 = pd.read_csv(f"{BASE_DIR}/463_unique_numerical_properties.csv")

physchem_num_all = pd.concat([physchem_norm, extra_463[physchem_num.columns]], ignore_index=True)

mutation_matrices = pd.read_csv(f"{BASE_DIR}/aaindex_square_diagonal_properties.csv", keep_default_na=False)
mutation_matrices.set_index("wild_mut", inplace=True)

# -----------------------------
# HELPER FUNCTIONS
# -----------------------------
def get_sequence_and_gene(uniprot_id: str):
    seq = next(iter(uniprot_seq.loc[uniprot_seq["uniprot_ids"] == uniprot_id, "Sequence"]))
    gene = next(iter(uniprot_seq.loc[uniprot_seq["uniprot_ids"] == uniprot_id, "gene"]))
    return seq, gene

def window_13mer(sequence: str, pos_1based: int) -> str:
    pos0 = pos_1based - 1
    left = max(0, pos0 - 6)
    right = min(len(sequence), pos0 + 7)
    return sequence[left:right]

def safe_get(series, default=0):
    try:
        return next(iter(series))
    except StopIteration:
        return default

def compute_dipeptide_motifs(sequence: str, pos_1based: int, wild: str):
    pos0 = pos_1based - 1
    n1 = sequence[pos0 - 1] if pos0 - 1 >= 0 else '-'
    n2 = sequence[pos0 - 2] if pos0 - 2 >= 0 else '-'
    n4 = sequence[pos0 - 4] if pos0 - 4 >= 0 else '-'
    c1 = sequence[pos0 + 1] if pos0 + 1 < len(sequence) else '-'
    c2 = sequence[pos0 + 2] if pos0 + 2 < len(sequence) else '-'
    c0 = sequence[pos0 + 0] if pos0 < len(sequence) else '-'

    return {
        "nM": n1 + wild, "Mc": wild + c0, "tri": n1 + wild + c0,
        "n_M": n2 + "*" + wild, "M_c": wild + "*" + c1,
        "n__M": (n4 + "**" + wild) if n4 != '-' else ('-' + wild),
        "M__c": wild + "**" + (c2 if c2 != '-' else c1)
    }

def add_physchem_features(dict_prop, wild, mut, win13):
    denom = sum(c.isalpha() for c in win13)
    a = sum(physchem_norm[win13[k]] for k in range(len(win13))) / denom
    b = sum(physchem_num_all[win13[k]] for k in range(len(win13))) / denom

    for idx, prop_name in enumerate(physchem_norm["index"]):
        w_norm = physchem_norm.loc[physchem_norm["index"] == prop_name, wild].values[0]
        m_norm = physchem_norm.loc[physchem_norm["index"] == prop_name, mut].values[0]
        dict_prop[f"{prop_name}_normalize_site_value"] = w_norm
        dict_prop[f"{prop_name}_normalize"] = a[idx]
        dict_prop[f"{prop_name}_normalize_diff"] = a[idx] - m_norm + w_norm

        w_num = physchem_num_all.loc[physchem_num_all["index"] == prop_name, wild].values[0]
        m_num = physchem_num_all.loc[physchem_num_all["index"] == prop_name, mut].values[0]
        dict_prop[f"{prop_name}_numeric_site_value"] = w_num
        dict_prop[f"{prop_name}_numeric_values"] = b[idx]
        dict_prop[f"{prop_name}_numeric_diff"] = b[idx] - m_num + w_num

    dict_prop.update({
        "neg_charge": len(re.findall("[DE]", win13)),
        "polar": len(re.findall("[NQSTP]", win13)),
        "aromatic": len(re.findall("[YFW]", win13)),
        "S_containing": len(re.findall("[CM]", win13)),
        "aliphatic": len(re.findall("[GALIV]", win13))
    })

def add_mutation_matrix_features(dict_prop, sequence, pos, wild, mut):
    pos0 = pos - 1
    prev_res = sequence[pos0 - 1] if pos0 - 1 >= 0 else "-"
    next_res = sequence[pos0 + 1] if pos0 + 1 < len(sequence) else "-"
    
    prop_n = mutation_matrices.loc[mut + prev_res] - mutation_matrices.loc[wild + prev_res]
    try:
        prop_c = mutation_matrices.loc[mut + next_res] - mutation_matrices.loc[wild + next_res]
    except KeyError:
        prop_c = mutation_matrices.loc["AG"] - mutation_matrices.loc["AG"]

    for col in prop_n.index:
        dict_prop[col] = prop_n[col]
        dict_prop[col.lower()] = prop_c[col]

def add_odds_ratio_features(dict_prop, motifs):
    for motif in MOTIFS:
        x = pd.read_excel(f"{BASE_DIR}/BRCA_odds_ratio.xlsx", sheet_name=motif)
        row = x[x[motif] == motifs[motif]]
        odd = safe_get(row["odd_ratio"], default=0)
        
        conditions = [(row["odd_ratio"] >= 1.2) | (row["odd_ratio"] == "inf"),
                      (row["odd_ratio"] < 1.2) & (row["odd_ratio"] >= 0.9),
                      (row["odd_ratio"] < 0.9)]
        dict_prop[f"{motif}_coded_odds"] = safe_get(np.select(conditions, [1, 3, 2]), default=0)
        dict_prop[f"{motif}_odds_ratio"] = odd

def add_pssm_features(dict_prop, f4_lines, pos, wild, mut):
    cols = f4_lines[pos].strip().split()
    aa_order = list("ARNDCQEGHILKMFPSTWYV")
    score_map = {k: int(v) for k, v in zip(aa_order, cols[2:22])}

    dict_prop["pssm_score1"] = score_map.get(wild, 0)
    dict_prop["pssm_score2"] = sum(score_map.values()) / 20.0
    dict_prop["pssm_score3"] = score_map.get(mut, 0) - score_map.get(wild, 0)
    
    cons = f4_lines[pos].strip("\n")[90:].split()[21:22]
    dict_prop["conservation"] = float(cons[0]) if len(cons) else 0.0

# -----------------------------
# MAIN LOOP
# -----------------------------
all_rows = []
for i in range(0, len(INPUT_MUTATIONS) - 1, 2):
    uniprot_id, mutation = INPUT_MUTATIONS[i], INPUT_MUTATIONS[i+1]
    wild, mut, pos = mutation[0], mutation[-1], int(mutation[1:-1])
    sequence, gene = get_sequence_and_gene(uniprot_id)

    if not (1 <= pos <= len(sequence)) or sequence[pos - 1] != wild:
        continue

    win13 = window_13mer(sequence, pos)
    motifs = compute_dipeptide_motifs(sequence, pos, wild)
    
    # Sequence-based auxiliary files
    f4_lines = open(f"{BASE_DIR}/{uniprot_id}.pssm", "r").readlines()[2:]
    f_aacon = pd.read_csv(f"{BASE_DIR}/{uniprot_id}.csv", skiprows=1, names=AACON_HEADER)

    dict_prop = {
        "UniProt ID": uniprot_id, "Gene Name": gene, "Mutation": mutation,
        "Wild": wild, "Mut": mut, "Pos": pos,
    }

    add_physchem_features(dict_prop, wild, mut, win13)
    add_mutation_matrix_features(dict_prop, sequence, pos, wild, mut)
    dict_prop.update(motifs)
    add_odds_ratio_features(dict_prop, motifs)
    add_pssm_features(dict_prop, f4_lines, pos, wild, mut)
    
    # Add AACon Features
    mm = f_aacon.iloc[pos - 1]
    for item in AACON_HEADER:
        dict_prop[item] = mm[item]

    all_rows.append(dict_prop)

pd.DataFrame(all_rows).to_csv(OUT_CSV, index=False)
print(f"Success! Sequence-only features saved to: {OUT_CSV}")